# 01 — Exploratory Data Analysis & Preprocessing

**Owner**: Noah  
**Course**: Machine Learning III (Unsupervised Learning) — Albert School  
**Dataset**: AI4I 2020 Predictive Maintenance (10 000 observations)

**Goal**: understand the data, identify cleaning needs, and design the preprocessing pipeline that will be consumed by the four anomaly detection models (Isolation Forest, One-Class SVM, LOF, Elliptic Envelope).

**Key constraint**: the column `Machine failure` (and its subtypes `TWF`, `HDF`, `PWF`, `OSF`, `RNF`) must NOT be used as a training signal. We only inspect the failure rate to inform the `contamination` hyperparameter, then set the labels aside for the Part 4 final evaluation.

## 1. Imports and configuration

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, str(Path.cwd().parent))

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

DATA_PATH = Path.cwd().parent / "ai4i2020.csv"

## 2. Load the dataset

We load the CSV and look at the first rows to confirm the schema.

In [ ]:
df = pd.read_csv(DATA_PATH)
df.head()

## 3. Schema and missing values

We expect 14 columns: 2 identifiers (`UDI`, `Product ID`), 1 categorical (`Type`), 5 numeric sensors, and 6 label columns (the global `Machine failure` flag plus 5 failure subtypes).

In [ ]:
df.info()

In [ ]:
missing = df.isna().sum()
print(missing)
print(f"\nTotal missing values: {missing.sum()}")

**Observation**: 10 000 rows × 14 columns, **zero missing values**. This is a clean industrial dataset, so we can skip imputation entirely. All numeric features are already in the right dtype (`float64` for temperatures and torque, `int64` for rotational speed and tool wear).

In [ ]:
df.describe()

## 4. Identify columns to keep, drop, and hold out

For unsupervised anomaly detection we must isolate three groups:

- **Identifiers** (`UDI`, `Product ID`): drop — no information for an anomaly model.
- **Held-out labels** (`Machine failure` + 5 failure subtypes `TWF`, `HDF`, `PWF`, `OSF`, `RNF`): drop from training. The 5 subtypes are essentially leaks of `Machine failure`. We keep `Machine failure` aside for the Part 4 "reveal".
- **Features** (`Type` + 5 numeric sensors): used for training.

In [ ]:
ID_COLUMNS = ["UDI", "Product ID"]
LABEL_COLUMNS = ["Machine failure", "TWF", "HDF", "PWF", "OSF", "RNF"]
NUMERIC_FEATURES = [
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]",
]
CATEGORICAL_FEATURES = ["Type"]

y_true = df["Machine failure"].copy()
X = df.drop(columns=ID_COLUMNS + LABEL_COLUMNS)

print(f"X shape: {X.shape}")
print(f"X columns: {list(X.columns)}")
print(f"y_true held out: {y_true.shape[0]} rows")

## 5. Distribution of numeric features

We plot histograms and boxplots side by side for each of the 5 sensor variables. We are looking for: skewness (which threatens the Gaussian assumption of Elliptic Envelope), outliers (which could be the anomalies we want to detect), and scale differences (which justify standardization for distance-based models).

In [ ]:
fig, axes = plt.subplots(len(NUMERIC_FEATURES), 2, figsize=(12, 3 * len(NUMERIC_FEATURES)))
for i, col in enumerate(NUMERIC_FEATURES):
    sns.histplot(df[col], kde=True, ax=axes[i, 0])
    axes[i, 0].set_title(f"Histogram — {col}")
    sns.boxplot(x=df[col], ax=axes[i, 1])
    axes[i, 1].set_title(f"Boxplot — {col}")
plt.tight_layout()
plt.savefig("../outputs/figures/numeric_distributions.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
skew_summary = df[NUMERIC_FEATURES].skew().round(3).rename("skewness").to_frame()
skew_summary["abs_skew"] = skew_summary["skewness"].abs()
skew_summary.sort_values("abs_skew", ascending=False)

**Takeaways from distributions**

- **Air temperature** and **Process temperature** look approximately symmetric (skew ≈ 0.1 and 0.0) — compatible with a Gaussian assumption.
- **Torque** is essentially symmetric (skew ≈ 0).
- **Tool wear** is uniform-like (skew ≈ 0), as expected for a wear-time variable that grows linearly.
- **Rotational speed** is strongly right-skewed (**skew ≈ 1.99**) with a long tail of high-speed values. **This violates the Gaussian assumption** of Elliptic Envelope and is a key fact for our modelling critique.
- The features live on very different scales: temperatures in the 295-305 K range, torque around 40 Nm, tool wear up to ~250 min, but rotational speed up to several thousand rpm. **Distance-based models will be dominated by `Rotational speed [rpm]` if we don't standardize.**

## 6. Distribution of the categorical feature `Type`

`Type` encodes the product variant: `L` = Low quality, `M` = Medium, `H` = High. We check the class balance and the failure rate per type — useful context for the business narrative.

In [ ]:
type_counts = df["Type"].value_counts()
type_share = df["Type"].value_counts(normalize=True).round(4)
type_summary = pd.concat([type_counts.rename("count"), type_share.rename("share")], axis=1)
type_summary

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
sns.countplot(x="Type", data=df, order=["L", "M", "H"], ax=ax)
ax.set_title("Distribution of product Type (L=Low, M=Medium, H=High)")
ax.set_ylabel("Count")
plt.tight_layout()
plt.savefig("../outputs/figures/type_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

**Observation**: 60 % of products are Low quality (`L`), 30 % Medium (`M`), 10 % High (`H`). Imbalanced but not extreme. For the distance-based models we will encode `Type` with one-hot encoding (see section 8 for the rationale).

## 7. Correlation analysis

We compute the Pearson correlation matrix on the 5 numeric features. We are looking for redundant pairs (which inflate the dimensionality of distance-based models) and physically meaningful relationships (which reassure us that the data is consistent).

In [ ]:
corr = df[NUMERIC_FEATURES].corr()
corr.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    corr,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    vmin=-1, vmax=1,
    center=0,
    square=True,
    ax=ax,
)
ax.set_title("Correlation matrix — numeric features")
plt.tight_layout()
plt.savefig("../outputs/figures/correlation_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

**Two strong correlations stand out**:

1. **Air temperature ↔ Process temperature: +0.876**. The process temperature is a function of the ambient air temperature plus the heat generated by the machine — this is a physical constraint, not noise. We keep both features (each carries a different signal: ambient vs. internal heat), but we are aware of the multicolinearity, which could destabilize the covariance estimation in Elliptic Envelope.

2. **Rotational speed ↔ Torque: -0.875**. Classic mechanical inverse relationship: at constant power $P = \omega \cdot \tau$, increasing the angular speed $\omega$ reduces the torque $\tau$ for a given load. Again, a real physical signal — we keep both features.

All other pairs are essentially uncorrelated (|r| < 0.03), meaning **Tool wear evolves independently** of the other sensors — interesting because tool wear is often a leading indicator of failures.

## 8. Held-out target — failure rate (information only, not used in training)

We briefly inspect `Machine failure` to **inform the `contamination` hyperparameter** of our anomaly detection models, then we set the column aside and do not use it again until Part 4. This is a deliberate, documented choice: a fully blind unsupervised pipeline would have to guess the contamination, but the assignment lets us peek at the rate to calibrate our models — we just must not feed the column as a label.

In [ ]:
n_failures = int(y_true.sum())
rate = float(y_true.mean())
print(f"Total failures in dataset: {n_failures} / {len(y_true)}")
print(f"Failure rate: {rate:.4f} ({rate*100:.2f}%)")

In [ ]:
subtypes = ["TWF", "HDF", "PWF", "OSF", "RNF"]
subtype_summary = pd.DataFrame({
    "count": df[subtypes].sum(),
    "rate_pct": (df[subtypes].mean() * 100).round(3),
})
subtype_summary.sort_values("count", ascending=False)

In [ ]:
failures_by_type = df.groupby("Type")["Machine failure"].agg(["count", "sum", "mean"]).round(4)
failures_by_type.columns = ["n_obs", "n_failures", "failure_rate"]
failures_by_type

**Takeaways**

- Overall failure rate: **3.39 %** (339 out of 10 000). This is the value we will use to seed the `contamination` hyperparameter in Part 2 — Isaac will sweep around it (e.g. 2 %, 3.4 %, 5 %, 10 %).
- Failure subtypes (in count): **HDF (115)**, **OSF (98)**, **PWF (95)**, **TWF (46)**, **RNF (19)**. These subtypes are highly informative for diagnosing *why* a machine fails, but they are leaks of `Machine failure` and must be excluded from training.
- Failure rate by product variant: **L (3.92 %) > M (2.77 %) > H (2.09 %)**. Low-quality variants fail more often — coherent and reassuring (the data has business signal). This may matter for the managerial recommendation if Diego wants to argue for variant-aware monitoring.

## 9. Preprocessing rationale — why standardize, why one-hot

Anomaly detection models in this assignment fall into two families with very different sensitivities to feature scaling and categorical encoding. Below, we justify each choice using the EDA above so that Isaac can pick consistent hyperparameters in Part 2.

### 9.1 Why standardize numeric features?

Section 5 showed that our 5 numeric sensors live on **very different scales** (temperatures in the 295-305 K range, rotational speed up to several thousand rpm). The four models react differently to this:

| Model | Sensitive to scale? | Why |
|---|---|---|
| **LOF** | **Yes** | Uses Euclidean distances between $k$ nearest neighbors. Without scaling, $d(x, x') \approx \sqrt{(\Delta\text{rpm})^2 + \dots}$ — rotational speed dominates the distance and the other features become invisible. Scaling puts every feature on the same numerical footing. |
| **One-Class SVM (RBF kernel)** | **Yes** | The kernel $k(x, x') = \exp(-\gamma \|x - x'\|^2)$ is built on Euclidean distance — same problem as LOF. |
| **Elliptic Envelope** | **In theory no, in practice yes** | The Mahalanobis distance $d_M(x, \mu) = \sqrt{(x-\mu)^T \Sigma^{-1} (x-\mu)}$ is mathematically scale-invariant because $\Sigma$ absorbs the units. But the robust covariance estimator (MCD) can be numerically unstable when features differ by 3+ orders of magnitude. We standardize for safety. |
| **Isolation Forest** | **No** | Splits on individual axes with random thresholds — invariant to monotone rescaling. We could feed unscaled data to IF, but to keep a single shared `X_processed` matrix across the four models, we apply the same `StandardScaler` to everyone (no harm done to IF). |

**Decision**: apply `StandardScaler` to all 5 numeric features. Document that IF does not require it.

### 9.2 How to encode the categorical feature `Type`?

`Type` has 3 levels: `L`, `M`, `H` (Low / Medium / High quality). Two encoding choices:

- **Ordinal encoding** (`L=0, M=1, H=2`) assumes a meaningful order *and* equal spacing. The labels are ordered (L < M < H in product quality), but **the metric is not equal-spaced**: nothing tells us that the gap between `L` and `M` is the same as between `M` and `H`. Forcing a 0/1/2 encoding would inject an arbitrary distance into Euclidean and Mahalanobis computations.

- **One-hot encoding** transforms `Type` into 3 binary columns (or 2 with `drop="first"`). No imposed metric: the distance between any two distinct types becomes a constant, which is honest given we have no quantitative scale.

**Decision**: one-hot encode `Type` with `drop="first"` to avoid perfect collinearity (`L+M+H=1`). Combined with `StandardScaler` on numeric features, this yields a `ColumnTransformer`-based pipeline implemented in `src/preprocessing.py`.